# Phase 4 — Delta Lake Deep Dive

**Duration:** Weeks 6–7 | **Goal:** Master Delta Lake operations for production data engineering.

---

### What We'll Cover

| Area | Key Skills |
| --- | --- |
| MERGE (Upserts) | Insert + Update in one atomic operation |
| UPDATE / DELETE | Row-level mutations on a lake |
| OPTIMIZE | File compaction for read performance |
| Z-ORDER | Co-locate related data for faster filters |
| Liquid Clustering | Next-gen clustering (replaces Z-ORDER) |
| VACUUM | Clean up old files, manage storage |
| Table Properties | Tuning Delta behavior |

### Dataset

* **Primary:** `samples.tpch.lineitem` (30M rows) — subsets for Delta operations
* **Volume:** `arnalladatabricks.pyspark_learning.raw_data` — Delta tables stored here
* **Managed tables:** `arnalladatabricks.pyspark_learning` schema — for Unity Catalog tables

---

**Pre-requisite:** Phase 3 completed. You should understand Delta basics (ACID, time travel, schema evolution).

In [0]:
# ============================================================
# STEP 1: Setup — load data and create our Delta playground
# ============================================================
from pyspark.sql.functions import col, lit, current_timestamp, when, count
from delta.tables import DeltaTable

# Volume and schema paths
VOLUME_PATH = "/Volumes/arnalladatabricks/pyspark_learning/raw_data"
SCHEMA = "arnalladatabricks.pyspark_learning"

# Load source data
df = spark.read.table("samples.tpch.lineitem")

# Create a manageable subset (100K rows) for Delta operations
df_subset = df.limit(100_000)

# Write as a managed Delta table we'll use throughout this phase
spark.sql(f"DROP TABLE IF EXISTS {SCHEMA}.lineitem_delta")
df_subset.write.saveAsTable(f"{SCHEMA}.lineitem_delta")

row_count = spark.table(f"{SCHEMA}.lineitem_delta").count()
print(f"✅ Created {SCHEMA}.lineitem_delta")
print(f"   Rows: {row_count:,}")
print(f"   Columns: {len(df_subset.columns)}")
print(f"\n   This is our playground for MERGE, UPDATE, DELETE, OPTIMIZE, etc.")

## 4.1 — MERGE (Upserts): The Most Important Delta Operation

### The Problem

Every day, your source system sends you data that contains:
* **New rows** (never seen before → INSERT)
* **Updated rows** (existing key, new values → UPDATE)
* **Sometimes deletes** (row removed from source → DELETE)

You can't just `append` (duplicates!) or `overwrite` (lose history!). You need **MERGE** — a single atomic operation that handles all three cases.

### The Syntax
```python
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "catalog.schema.table")

target.alias("t").merge(
    source_df.alias("s"),
    condition="t.id = s.id"           # How to match rows
).whenMatchedUpdate(
    set={"col1": "s.col1", "col2": "s.col2"}  # Update existing
).whenNotMatchedInsertAll()            # Insert new rows
.execute()
```

### The Three Clauses

| Clause | Fires when | Real-world meaning |
| --- | --- | --- |
| `whenMatchedUpdate` | Source key EXISTS in target | "Update this customer's address" |
| `whenMatchedDelete` | Source key EXISTS + condition | "Customer requested deletion" |
| `whenNotMatchedInsert` | Source key NOT in target | "New customer, add them" |
| `whenNotMatchedBySourceDelete` | Target key NOT in source | "Customer disappeared from source" |

In [0]:
# ============================================================
# STEP 2: MERGE — upsert (insert + update) in one operation
# ============================================================
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, current_timestamp

# --- Create a small "orders" table to demo MERGE clearly ---
spark.sql(f"DROP TABLE IF EXISTS {SCHEMA}.customers")

customers_initial = spark.createDataFrame([
    (1, "Alice", "alice@email.com", "New York", 50000.00),
    (2, "Bob", "bob@email.com", "Chicago", 60000.00),
    (3, "Charlie", "charlie@email.com", "Boston", 55000.00),
    (4, "Diana", "diana@email.com", "Seattle", 70000.00),
    (5, "Eve", "eve@email.com", "Austin", 65000.00),
], ["id", "name", "email", "city", "salary"])

customers_initial.write.saveAsTable(f"{SCHEMA}.customers")
print("BEFORE MERGE — Current customers table:")
spark.table(f"{SCHEMA}.customers").show()

# --- Simulate an "incoming update" from source system ---
# Contains: 2 updates (Alice moved, Bob got a raise) + 2 new customers
incoming_data = spark.createDataFrame([
    (1, "Alice", "alice@email.com", "San Francisco", 55000.00),  # UPDATE: moved city + raise
    (2, "Bob", "bob@email.com", "Chicago", 75000.00),            # UPDATE: salary change
    (6, "Frank", "frank@email.com", "Denver", 62000.00),         # NEW
    (7, "Grace", "grace@email.com", "Portland", 58000.00),       # NEW
], ["id", "name", "email", "city", "salary"])

print("Incoming data (2 updates + 2 new):")
incoming_data.show()

# --- MERGE: match on id, update if exists, insert if new ---
target = DeltaTable.forName(spark, f"{SCHEMA}.customers")

target.alias("t").merge(
    incoming_data.alias("s"),
    condition="t.id = s.id"
).whenMatchedUpdateAll(    # Update ALL columns when key matches
).whenNotMatchedInsertAll( # Insert ALL columns when key doesn't exist
).execute()

print("AFTER MERGE — Table now has updates + new rows:")
df_merged = spark.table(f"{SCHEMA}.customers").orderBy("id")
df_merged.show()
print(f"Rows: 5 → {df_merged.count()} (2 updated in place, 2 inserted)")

### Deep Dive: How MERGE Works Under the Hood

**What actually happens when you run MERGE:**

1. Spark reads the source DataFrame and the target Delta table
2. JOIN them on your condition (`t.id = s.id`)
3. For each row in the join result:
   - Match found → apply `whenMatchedUpdate` (or delete)
   - No match → apply `whenNotMatchedInsert`
4. Write NEW Parquet files with the results
5. Update the Delta transaction log (atomically!)
6. Old files are NOT deleted yet (time travel still works)

**`whenMatchedUpdateAll` vs `whenMatchedUpdate(set={...})`:**

| Method | When to use |
| --- | --- |
| `whenMatchedUpdateAll()` | Source has the same columns as target — update everything |
| `whenMatchedUpdate(set={"city": "s.city"})` | Only update specific columns (leave others untouched) |

**Real-world MERGE patterns:**

* **SCD Type 1** (overwrite): `whenMatchedUpdateAll()` — just replace old values
* **SCD Type 2** (history): Mark old row as inactive, insert new row with current flag
* **Deduplification**: MERGE with `whenMatchedUpdate` doing nothing, `whenNotMatchedInsert` — effectively "insert if not exists"
* **Delete propagation**: `whenMatchedDelete()` with a condition like `s.is_deleted = true`

**Performance tip:** MERGE does a JOIN. If your target table is huge (billions of rows), add partition filters or Z-ORDER on the join key to make it fast.

## 4.2 — UPDATE & DELETE: Row-Level Mutations

Delta Lake lets you update or delete specific rows — something impossible with plain Parquet.

### UPDATE
```python
# SQL style
spark.sql("UPDATE schema.table SET col = value WHERE condition")

# Python style
dt = DeltaTable.forName(spark, "schema.table")
dt.update(
    condition="id = 1",
    set={"city": lit("Miami")}
)
```

### DELETE
```python
# SQL style
spark.sql("DELETE FROM schema.table WHERE condition")

# Python style
dt.delete(condition="salary < 50000")
```

### How it works (Copy-on-Write)

Delta doesn't modify files in place. Instead:
1. Identifies which files contain affected rows
2. Reads those files
3. Writes NEW files with the changes applied
4. Updates the transaction log to point to the new files
5. Old files remain (for time travel) until VACUUM

This is why DELETE/UPDATE on a 1-row change still rewrites entire files. It's fast for bulk operations, but not designed for single-row OLTP workloads.

In [0]:
# ============================================================
# STEP 3: UPDATE and DELETE — row-level mutations
# ============================================================
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, upper

dt = DeltaTable.forName(spark, f"{SCHEMA}.customers")

print("BEFORE mutations:")
spark.table(f"{SCHEMA}.customers").orderBy("id").show()

# --- UPDATE: Give everyone in Chicago a 10% raise ---
print("UPDATE: 10% raise for Chicago employees...")
dt.update(
    condition=col("city") == "Chicago",
    set={"salary": col("salary") * 1.10}
)
spark.table(f"{SCHEMA}.customers").filter(col("city") == "Chicago").show()

# --- UPDATE: Uppercase all emails ---
print("UPDATE: Uppercase all city names...")
dt.update(
    set={"city": upper(col("city"))}
)

# --- DELETE: Remove employees with salary < 56000 ---
print("DELETE: Remove rows where salary < 56000...")
before_count = spark.table(f"{SCHEMA}.customers").count()
dt.delete(condition=col("salary") < 56000)
after_count = spark.table(f"{SCHEMA}.customers").count()

print(f"  Rows before: {before_count}")
print(f"  Rows after:  {after_count}")
print(f"  Deleted:     {before_count - after_count} rows")

print("\nFINAL STATE:")
spark.table(f"{SCHEMA}.customers").orderBy("id").show()

# --- Show that time travel still has the old data ---
print("TIME TRAVEL: Version 0 (original 5 rows) still accessible:")
spark.sql(f"SELECT * FROM {SCHEMA}.customers VERSION AS OF 0").show()

### Deep Dive: Copy-on-Write vs Deletion Vectors

**The OLD way: Copy-on-Write (CoW)**

Parquet files are immutable (write-once). You can’t “edit” a byte inside a Parquet file. So Delta must:
1. Read the entire file containing the row
2. Apply the change in memory
3. Write a NEW file with the modification
4. Update the log: “remove old file, add new file”

**Example:** Your table has 100 files, each with 1M rows. You delete 1 row:
- Only 1 file contains that row
- That 1 file gets rewritten (999,999 rows in the new file)
- The other 99 files are untouched
- Total I/O: read 1M rows + write 999,999 rows

---

**The NEW way: Deletion Vectors (DVs)**

Instead of rewriting the entire file, Delta writes a tiny **bitmap file** alongside the data:

```
/table/
  part-00001.parquet           ← 1 million rows (UNTOUCHED)
  deletion_vector_00001.bin    ← tiny file: "rows 4532, 8901 are invalid"
```

The bitmap is NOT a “small Parquet file.” It’s a compact binary (roaring bitmap) that says: “in file part-00001.parquet, skip row positions X, Y, Z.” Readers check the bitmap and skip those rows.

**DVs work for BOTH delete AND update:**

| Operation | What DV does |
| --- | --- |
| **DELETE** | Mark row #4532 as deleted in the DV bitmap. Done. No file rewrite. |
| **UPDATE** | Mark row #4532 as deleted in the DV bitmap + write a tiny NEW file containing only the updated row. |

An UPDATE is really: “delete the old version” + “insert the new version.” DVs make the “delete” part instant.

**Performance comparison:**

| | Copy-on-Write (old) | Deletion Vectors (new) |
| --- | --- | --- |
| Delete 1 row from 1M-row file | Rewrite 999,999 rows | Write ~100 byte bitmap |
| Update 1 row | Rewrite 999,999 rows | Write bitmap + 1 tiny file |
| Bulk update 50% of rows | Similar cost | Similar cost (DVs don’t help much) |

**What happens to DVs over time?**

After many small deletes/updates, you accumulate:
- Original large files (untouched)
- Many small DV bitmaps
- Many tiny “update insert” files

Running **OPTIMIZE** cleans this up — it materializes all pending DVs by rewriting files without the deleted rows and merging the tiny update files. Clean slate.

```
BEFORE OPTIMIZE:  big_file.parquet + deletion_vector.bin + tiny_update1.parquet
AFTER OPTIMIZE:   clean_file.parquet (all changes applied, DVs gone)
```

**Are DVs enabled by default?** Yes, on Databricks (DBR 14.0+). Check with:
```sql
SHOW TBLPROPERTIES schema.table ('delta.enableDeletionVectors')
```

---

**When to use UPDATE/DELETE vs MERGE:**

| Operation | Use when |
| --- | --- |
| `UPDATE` | You know exactly which rows to change (e.g., fix a data quality issue) |
| `DELETE` | Remove specific rows (GDPR deletion, remove test data) |
| `MERGE` | Incoming batch with mixed inserts/updates/deletes |

## 4.3 — OPTIMIZE & Z-ORDER: File Compaction and Data Layout

### The Small Files Problem

After many appends, updates, and merges, your Delta table accumulates hundreds of tiny files. Reading 1000 small files is MUCH slower than reading 10 large files because:
* Each file open = metadata overhead
* Small files can't leverage sequential I/O
* More tasks to schedule = more overhead

### OPTIMIZE: Compact small files into large ones
```sql
OPTIMIZE catalog.schema.table_name
```
This reads all small files and rewrites them as fewer, larger files (~1 GB each). Data content doesn't change — just file layout.

### Z-ORDER: Co-locate related data physically
```sql
OPTIMIZE catalog.schema.table_name ZORDER BY (column1, column2)
```
After compaction, Z-ORDER sorts data so rows with similar values in your chosen columns end up in the same files. This enables **data skipping** — when you filter on those columns, Spark can skip entire files.

### When to Z-ORDER:

| Good Z-ORDER columns | Bad Z-ORDER columns |
| --- | --- |
| Columns in WHERE clauses | Columns never filtered on |
| High cardinality (customer_id, date) | Low cardinality (boolean, status) |
| Join keys | Columns you only SELECT (never filter) |
| Max 2-4 columns | More than 4 (diminishing returns) |

In [0]:
# ============================================================
# STEP 4: OPTIMIZE and Z-ORDER
# ============================================================
from pyspark.sql.functions import col

# --- Create a table with many small files (simulate many appends) ---
spark.sql(f"DROP TABLE IF EXISTS {SCHEMA}.orders_fragmented")

# Write in 20 small batches to simulate daily appends
df_source = df.select("l_orderkey", "l_shipmode", "l_shipdate", "l_extendedprice").limit(100_000)

# First batch
df_source.limit(5_000).write.saveAsTable(f"{SCHEMA}.orders_fragmented")

# Append 19 more batches (creates many small files)
for i in range(19):
    df_source.limit(5_000).write.mode("append").saveAsTable(f"{SCHEMA}.orders_fragmented")

# Check file count BEFORE optimize
detail_before = spark.sql(f"DESCRIBE DETAIL {SCHEMA}.orders_fragmented").select("numFiles").collect()[0][0]
print(f"BEFORE OPTIMIZE:")
print(f"  Files: {detail_before}")
print(f"  Rows:  {spark.table(f'{SCHEMA}.orders_fragmented').count():,}")

# --- OPTIMIZE: Compact files ---
print("\nRunning OPTIMIZE...")
result = spark.sql(f"OPTIMIZE {SCHEMA}.orders_fragmented")
result.show(truncate=False)

detail_after = spark.sql(f"DESCRIBE DETAIL {SCHEMA}.orders_fragmented").select("numFiles").collect()[0][0]
print(f"AFTER OPTIMIZE:")
print(f"  Files: {detail_after} (compacted from {detail_before}!)")

# --- Z-ORDER: Sort data for faster filtering ---
print("\nRunning OPTIMIZE with Z-ORDER BY (l_shipdate, l_shipmode)...")
spark.sql(f"OPTIMIZE {SCHEMA}.orders_fragmented ZORDER BY (l_shipdate, l_shipmode)")

# Demo: data skipping in action
print("\nDATA SKIPPING DEMO:")
print("Query: WHERE l_shipdate = '1995-06-15'")
spark.table(f"{SCHEMA}.orders_fragmented") \
    .filter(col("l_shipdate") == "1995-06-15") \
    .select("l_orderkey", "l_shipdate", "l_shipmode", "l_extendedprice") \
    .show(5)
print("→ With Z-ORDER, Spark skips files that don't contain 1995-06-15!")

### Deep Dive: How Data Skipping Works

**Without Z-ORDER:**

Files contain random mixes of dates. To find `shipdate = '1995-06-15'`, Spark must scan ALL files (each might contain any date).

**With Z-ORDER on shipdate:**

Files are organized so similar dates cluster together:
* File 1: dates 1992-01 to 1992-06
* File 2: dates 1992-07 to 1993-01
* File 3: dates 1993-02 to 1993-08
* ...

Now `WHERE shipdate = '1995-06-15'` only needs to read 1-2 files. The rest are **skipped** based on min/max stats in the Delta log.

**Z-ORDER vs regular sorting:**

Regular `ORDER BY` sorts on one dimension. Z-ORDER uses a **space-filling curve** that interleaves bits from multiple columns, so data is co-located across ALL Z-ORDER columns simultaneously. This is why it works for multi-column filters.

**Predictive Optimization (Databricks feature):**

In production, you don't manually run OPTIMIZE. Enable **Predictive Optimization** and Databricks automatically:
* Runs OPTIMIZE when small files accumulate
* Runs VACUUM when old files pile up
* Chooses when to Z-ORDER based on query patterns

```sql
ALTER TABLE schema.table SET TBLPROPERTIES ('delta.enableOptimizeWrite' = 'true');
```

**`optimizeWrite`** = each write produces optimally-sized files (prevents small file problem at the source).

## 4.4 — Liquid Clustering (Next-Gen, Replaces Z-ORDER)

### What's wrong with Z-ORDER?

* You must run `OPTIMIZE ZORDER BY` manually (or schedule it)
* Rewrites ALL data every time (even unchanged files)
* Can't change clustering columns without rewriting everything
* Expensive for large tables

### Liquid Clustering = Automatic, Incremental, Flexible

```sql
-- Create a table with liquid clustering
CREATE TABLE schema.table (...) CLUSTER BY (col1, col2)

-- Change clustering columns later (no rewrite!)
ALTER TABLE schema.table CLUSTER BY (col3, col4)

-- Trigger clustering (or let Predictive Optimization do it)
OPTIMIZE schema.table
```

### Key Differences

| Feature | Z-ORDER | Liquid Clustering |
| --- | --- | --- |
| Setup | Manual OPTIMIZE ZORDER BY | `CLUSTER BY` at table creation |
| Incremental | No (rewrites all) | Yes (only new/changed files) |
| Change columns | Requires full rewrite | `ALTER TABLE CLUSTER BY` (instant) |
| Runs automatically | Only with Predictive Opt | Yes, with OPTIMIZE or Predictive Opt |
| Interacts with partitioning | Works alongside partitionBy | **Replaces** partitioning entirely |

### When to Use What

* **New tables:** Always use Liquid Clustering
* **Existing Z-ORDER tables:** Migrate when convenient
* **Partitioned tables:** Consider replacing with Liquid Clustering if partition columns aren't ideal

In [0]:
# ============================================================
# STEP 5: Liquid Clustering
# ============================================================

# --- Create a table WITH liquid clustering ---
spark.sql(f"DROP TABLE IF EXISTS {SCHEMA}.orders_clustered")

spark.sql(f"""
    CREATE TABLE {SCHEMA}.orders_clustered (
        l_orderkey LONG,
        l_shipmode STRING,
        l_shipdate DATE,
        l_extendedprice DECIMAL(18,2),
        l_returnflag STRING
    )
    CLUSTER BY (l_shipdate, l_shipmode)
""")

print("✅ Created table with CLUSTER BY (l_shipdate, l_shipmode)")
print("   Liquid clustering is now active on this table.\n")

# Insert data
df_insert = df.select(
    "l_orderkey", "l_shipmode", "l_shipdate", "l_extendedprice", "l_returnflag"
).limit(200_000)

df_insert.write.mode("append").saveAsTable(f"{SCHEMA}.orders_clustered")
print(f"Inserted {spark.table(f'{SCHEMA}.orders_clustered').count():,} rows")

# Run OPTIMIZE (triggers liquid clustering)
print("\nRunning OPTIMIZE (triggers liquid clustering)...")
result = spark.sql(f"OPTIMIZE {SCHEMA}.orders_clustered")
result.show(truncate=False)

# --- Change clustering columns on the fly! ---
print("Changing clustering to (l_returnflag, l_shipdate)...")
spark.sql(f"ALTER TABLE {SCHEMA}.orders_clustered CLUSTER BY (l_returnflag, l_shipdate)")
print("✅ Done! No data rewrite needed — takes effect on next OPTIMIZE.")

# Show table properties
print("\nTable properties:")
spark.sql(f"DESCRIBE DETAIL {SCHEMA}.orders_clustered").select(
    "name", "numFiles", "sizeInBytes", "clusteringColumns"
).show(truncate=False)

## 4.5 — VACUUM: Storage Cleanup

### The Problem

Every MERGE, UPDATE, DELETE, and OPTIMIZE creates NEW files but keeps the OLD files (for time travel). Over time, storage grows unbounded.

### VACUUM removes old files
```sql
VACUUM schema.table RETAIN 168 HOURS   -- Keep 7 days of history
VACUUM schema.table RETAIN 0 HOURS     -- Delete everything (DANGEROUS)
```

**Default retention: 7 days (168 hours).** Files older than this that aren't referenced by the current table version get deleted.

### What VACUUM deletes:
* Old Parquet files from before UPDATE/DELETE/OPTIMIZE
* These are the files that enable time travel

### What happens AFTER vacuum:
* Time travel to versions before the vacuum → **FAILS** (files are gone)
* Current table → unaffected
* Recent versions (within retention) → still work

### Safety Rules

| Setting | Default | Meaning |
| --- | --- | --- |
| `delta.deletedFileRetentionDuration` | 7 days | How long to keep old files |
| `spark.databricks.delta.retentionDurationCheck.enabled` | true | Prevents `RETAIN 0 HOURS` (safety net) |

**Never set retention < 7 days in production** unless you're absolutely sure no running queries or streaming jobs reference old versions.

In [0]:
# ============================================================
# STEP 6: VACUUM — see how it affects time travel
# ============================================================

# Check the history of our customers table (has multiple versions from MERGE/UPDATE/DELETE)
print("HISTORY of customers table:")
history = spark.sql(f"DESCRIBE HISTORY {SCHEMA}.customers")
history.select("version", "timestamp", "operation").show(truncate=False)

# Show storage growth
print("\nSTORAGE DETAIL:")
detail = spark.sql(f"DESCRIBE DETAIL {SCHEMA}.customers")
detail.select("numFiles", "sizeInBytes").show()

# --- DRY RUN: See what VACUUM would delete ---
print("DRY RUN — what would VACUUM delete (files older than 0 hours):")
print("(We'll use RETAIN 0 to show all deletable files, but WON'T actually run it)")

# Show that time travel works BEFORE vacuum
print("\nTime travel BEFORE vacuum:")
v0_count = spark.sql(f"SELECT count(*) as cnt FROM {SCHEMA}.customers VERSION AS OF 0").collect()[0][0]
latest_count = spark.table(f"{SCHEMA}.customers").count()
print(f"  Version 0 (original): {v0_count} rows")
print(f"  Latest version:       {latest_count} rows")
print("  ✅ Both accessible!")

# Run VACUUM with default retention (7 days)
# Since all our versions were created today, nothing will be deleted
print("\nRunning VACUUM with default 168-hour retention:")
vacuum_result = spark.sql(f"VACUUM {SCHEMA}.customers RETAIN 168 HOURS")
print("✅ VACUUM complete (nothing deleted — all files are < 7 days old)")

print("\n" + "="*60)
print("KEY TAKEAWAY:")
print("  In production, VACUUM runs on a schedule (daily/weekly).")
print("  It keeps the last 7 days of history and deletes the rest.")
print("  After VACUUM, old time travel queries FAIL (files gone).")
print("="*60)

### Deep Dive: Table Properties & Production Settings

**Essential Delta table properties:**

| Property | What it does | Recommended |
| --- | --- | --- |
| `delta.autoOptimize.optimizeWrite` | Write optimally-sized files automatically | `true` |
| `delta.autoOptimize.autoCompact` | Auto-compact small files after writes | `true` |
| `delta.deletedFileRetentionDuration` | How long VACUUM keeps old files | `interval 7 days` |
| `delta.logRetentionDuration` | How long to keep transaction log entries | `interval 30 days` |
| `delta.enableChangeDataFeed` | Track row-level changes (CDC) | `true` for downstream consumers |

**Setting properties:**
```sql
ALTER TABLE schema.table SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);
```

**Production checklist for Delta tables:**
1. ✅ Enable `optimizeWrite` + `autoCompact` (prevents small files)
2. ✅ Set `CLUSTER BY` on columns used in WHERE/JOIN (liquid clustering)
3. ✅ Schedule VACUUM (or use Predictive Optimization)
4. ✅ Enable Change Data Feed if downstream systems need row-level diffs
5. ✅ Set appropriate retention based on your time travel needs

**DESCRIBE DETAIL — your table health check:**
```sql
DESCRIBE DETAIL schema.table
-- Shows: numFiles, sizeInBytes, partitionColumns, clusteringColumns, properties
```

In [0]:
# ============================================================
# STEP 7: Table properties — production settings
# ============================================================

# --- Set production-ready properties on our table ---
print("Setting production table properties...")
spark.sql(f"""
    ALTER TABLE {SCHEMA}.orders_clustered SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true',
        'delta.deletedFileRetentionDuration' = 'interval 7 days',
        'delta.logRetentionDuration' = 'interval 30 days',
        'delta.enableChangeDataFeed' = 'true'
    )
""")
print("✅ Properties set!\n")

# --- View all properties ---
print("TABLE PROPERTIES:")
props = spark.sql(f"SHOW TBLPROPERTIES {SCHEMA}.orders_clustered")
props.filter(col("key").startswith("delta.")).show(truncate=False)

# --- DESCRIBE DETAIL: the health check ---
print("\nTABLE DETAIL (health check):")
spark.sql(f"DESCRIBE DETAIL {SCHEMA}.orders_clustered").select(
    "format", "numFiles", "sizeInBytes", "clusteringColumns"
).show(truncate=False)

# --- Change Data Feed: see what changed ---
print("CHANGE DATA FEED — track what changed between versions:")
# Make a change so we can see the CDF
dt = DeltaTable.forName(spark, f"{SCHEMA}.orders_clustered")
dt.update(
    condition=col("l_shipmode") == "AIR",
    set={"l_extendedprice": col("l_extendedprice") * 1.05}  # 5% price increase for AIR
)

# Read the changes using SQL table_changes() function
latest_version = spark.sql(f"DESCRIBE HISTORY {SCHEMA}.orders_clustered").select("version").collect()[0][0]

print(f"Reading Change Data Feed for version {latest_version}:")
cdf = spark.sql(f"""
    SELECT l_orderkey, l_shipmode, l_extendedprice, _change_type, _commit_version
    FROM table_changes('{SCHEMA}.orders_clustered', {latest_version})
    WHERE l_shipmode = 'AIR'
    LIMIT 10
""")
cdf.show()
print("→ _change_type shows: 'update_preimage' (old value) and 'update_postimage' (new value)")
print("  This is how downstream systems detect exactly what changed!")

## Phase 4 Checkpoint

**Can you do these?**

- [ ] Write a MERGE that handles inserts AND updates in one operation
- [ ] Explain `whenMatchedUpdate` vs `whenNotMatchedInsert` vs `whenMatchedDelete`
- [ ] Use UPDATE/DELETE with conditions on a Delta table
- [ ] Explain copy-on-write (why Delta rewrites files for mutations)
- [ ] Run OPTIMIZE to compact small files and explain why it matters
- [ ] Use Z-ORDER and explain data skipping with min/max stats
- [ ] Create a table with Liquid Clustering and change columns on the fly
- [ ] Run VACUUM and explain its relationship to time travel
- [ ] Set production table properties (optimizeWrite, autoCompact, CDF)
- [ ] Use DESCRIBE DETAIL and DESCRIBE HISTORY for table health checks

---

**Key takeaways:**
1. **MERGE** is the workhorse of data engineering — handles mixed inserts/updates/deletes atomically
2. **OPTIMIZE** prevents the small files problem; **Z-ORDER** enables data skipping
3. **Liquid Clustering** replaces both partitioning AND Z-ORDER for new tables
4. **VACUUM** reclaims storage but kills time travel for old versions
5. **Table properties** (`optimizeWrite`, `autoCompact`, CDF) should be set on every production table

---

**Next up:** Phase 5 — Performance Tuning (shuffle tuning, caching, broadcast hints, skew handling, Spark UI)